## Patient Journey - Rx (NBA 3.0 baseline, frozen copy)

**Clone created 2026-10-29** from `03_patient_journey_rx` (the 3.0 cells), kept unchanged as the baseline for the NBA 3.1 work. Only change: the cohort config path is `../../00_cohort_config` (the notebook moved to `Rx Files`). The NBA 3.1 steps are built in `03_patient_journey_rx`; plan in `NBA_3_1_Rx_Journey_Plan.md`.

---

#### Data source used
* PHCDW.PHCDW_DSAA.IQV_LAAD_RX_VW
* PHCDW.PHCDW_DSAA.CKD_PRODUCT_MAPPING_VW
* PHCDW.PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PRODUCT
* PHCDW_STG_CAR.STG_IQVIA_LAAD_DIM_PROVIDER

#### Steps:
1. Query rx claims data after 2017-01-01 for patient who visited neph after 2018-10-01
2. Get date level medication list (with 7-day additional window for gaps between refills) for each patient
3. Save table to databricks

#### Notes:
1. Treatment Cateogry and Brand

|TREATMENT_CATEGORY            |BRAND                         |
|------------------------------|------------------------------|
|ACE/ ARB                      |ACE/ ARB                      |
|Calcium channel / Beta blocker|Calcium channel / Beta blocker|
|Combo drug                    |Combo drug                    |
|Diuretic_NON_MRA              |Diuretic_NON_MRA              |
|Insulin                       |Insulin                       |
|metformin                     |metformin                     |
|MRA                           |MRA                           |
|SGLT2                         |INVOKANA 300MG                |
|SGLT2                         |FARXIGA                       |
|SGLT2                         |SGLT2                         |
|SGLT2                         |INVOKANA 100MG                |
|SGLT2                         |JARDIANCE                     |

* when PROD.PRODUCT_NAME like '%INVOKANA%' and  PRODUCT_STRENGTH = '100 MG' then 'INVOKANA 100MG'
* when PROD.PRODUCT_NAME like '%INVOKANA%' and  PRODUCT_STRENGTH = '300 MG' then 'INVOKANA 300MG'
* when PROD.PRODUCT_NAME like '%JARDIANCE%' then 'JARDIANCE'
* when PROD.PRODUCT_NAME like '%FARXIGA%' then 'FARXIGA'

## 1. Setup Snowflake connection and load data

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "../../00_cohort_config"

## 2. Data cleanup and processing

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [0]:
# ===============================
# Step 1: Load / Assume tables as DataFrames
# ===============================
fact_rx_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_FCT_RX")
demographic_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PTNT_DMGRPHC")
provider_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRVDR")
product_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PRD")
plan_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_STG_CAR.STG_CAR_IQV_LAAD_DIM_PLN")

nba_hcp_df = spark.read.format("delta").load("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/kerendia_nba_3_hcp_model_base_hcp_universe_for_inference").select('PROVIDER_ID').distinct()

ckd_mapping_df = Get_Data_Snowflakes("SELECT * FROM PHCDW.PHCDW_DSAA.CKD_PRODUCT_MAPPING_VW")
print("Shape :", fact_rx_df.count(), len(fact_rx_df.columns))
print("Shape :", demographic_df.count(), len(demographic_df.columns))
print("Shape :", provider_df.count(), len(provider_df.columns))
print("Shape :", product_df.count(), len(product_df.columns))
print("Shape :", plan_df.count(), len(plan_df.columns))
print("Shape :", nba_hcp_df.count(), len(nba_hcp_df.columns))
print("Shape :", ckd_mapping_df.count(), len(ckd_mapping_df.columns))

In [0]:
# ===============================
# Step 2: Filter Fact RX & Join Dimensions
# ===============================
rx_base = (
    fact_rx_df.alias("f")
    .join(nba_hcp_df.alias("hcp"), F.col("f.PROVIDER_ID") == F.col("hcp.PROVIDER_ID"))
    .join(provider_df.alias("p"), F.col("f.PROVIDER_ID") == F.col("p.PROVIDER_ID"))
    .join(product_df.alias("prd"), F.col("f.NDC_CD") == F.col("prd.NDC_CD"))
    .join(plan_df.alias("pl"), F.col("f.PAYER_PLAN_ID") == F.col("pl.PAYER_PLAN_ID"))
    .join(demographic_df.alias("d"), F.col("f.PATIENT_ID") == F.col("d.PATIENT_ID"), "left")
    .join(ckd_mapping_df.alias("map"), F.col("prd.PRODUCT_NAME") == F.col("map.PRODUCT_NAME"))
    .filter(
        (F.col("f.SVC_DT") >= F.lit(DATA_START_DATE)) &
        (F.col("p.NPI_NUMBER").isNotNull()) &
        (~F.lower(F.col("map.Treatment_Category")).like("%other%"))
    )
    .select(
        "f.PATIENT_ID",
        "f.SVC_DT",
        "f.DAYS_SUPPLY",
        "prd.PRODUCT_NAME",
        "prd.PRODUCT_STRENGTH",
        "map.Treatment_Category",
        "p.NPI_NUMBER",
        "p.PRIMARY_SPECIALTY_DESC"
    )
)
# Validation filter: 10 test HCPs (from config)
rx_base, test_patients = apply_hcp_filter(rx_base)

print("Shape :", rx_base.count(), len(rx_base.columns))
rx_base.limit(100).display()

In [0]:
# ===============================
# Step 3: Compute END_DT per row
# ===============================
max_date = fact_rx_df.agg(F.max("SVC_DT").alias("max_date")).collect()[0]["max_date"]

rx_enriched = (
    rx_base
    .withColumn("DAYS_SUPPLY_T", F.round(F.col("DAYS_SUPPLY")).cast("int"))
    .withColumn("END_DT", F.least(
        F.expr("date_add(SVC_DT, DAYS_SUPPLY_T + 7)"),
        F.lit(max_date)
    ))
)
print("Shape :", rx_enriched.count(), len(rx_enriched.columns))
print("Max Date:", max_date)
rx_enriched.filter(F.col("PATIENT_ID") == "590252630").display()

In [0]:
# ===============================
# Step 4: Map Category using Spark SQL expressions (vectorized)
# ===============================
rx_enriched = rx_enriched.withColumn(
    "CATEGORY",
    F.when((F.col("PRODUCT_NAME").like("%INVOKANA%")) & (F.col("PRODUCT_STRENGTH") == "100 MG"), "INVOKANA 100MG")
     .when((F.col("PRODUCT_NAME").like("%INVOKANA%")) & (F.col("PRODUCT_STRENGTH") == "300 MG"), "INVOKANA 300MG")
     .when(F.col("PRODUCT_NAME").like("%JARDIANCE%"), "JARDIANCE")
     .when(F.col("PRODUCT_NAME").like("%FARXIGA%"), "FARXIGA")
     .when(F.col("PRODUCT_NAME").isin(
         'ALOGLIPTIN BENZOATE','ALOGLIPTIN-METFORMIN HCL','ALOGLIPTIN-PIOGLITAZONE',
         'SITAGLIPTIN-METFORMIN HCL','SITAGLIPTIN PHOSPHATE','LINAGLIPTIN-METFORMIN HCL',
         'SAXAGLIPTIN-METFORMIN HCL','SAXAGLIPTIN HCL','LINAGLIPTIN','ALOGLIPTIN',
         'ALOGLIPTIN/METFORMIN HCL','ALOGLIPTIN/PIOGLITAZONE','JANUMET','JANUMET XR',
         'JANUVIA','JENTADUETO','JENTADUETO XR','KAZANO','KOMBIGLYZE XR','NESINA',
         'ONGLYZA','OSENI','TRADJENTA'
     ), "DPP-IV")
     .when(F.col("PRODUCT_NAME").isin(
         'LIXISENATIDE','EXENATIDE','SEMAGLUTIDE','ALBIGLUTIDE','DULAGLUTIDE',
         'LIRAGLUTIDE','INSULIN DEGLUDEC-LIRAGLUTIDE','ADLYXIN','ADLYXIN STARTER PACK',
         'BYDUREON','BYDUREON BCISE','BYDUREON PEN','BYETTA','OZEMPIC','RYBELSUS',
         'TANZEUM','TRULICITY','VICTOZA','XULTOPHY 100/3.6'
     ), "GLP-1")
     .otherwise(F.col("Treatment_Category"))
)
print("Shape :", rx_enriched.count(), len(rx_enriched.columns))
rx_enriched.filter(F.col("PATIENT_ID") == "590252630").display()

In [0]:
# ===============================
# Step 5: Expand dates per day using explode(sequence())
# ===============================
rx_expanded = rx_enriched.withColumn(
    "TXN_DATE",
    F.explode(F.expr("sequence(SVC_DT, END_DT, interval 1 day)"))
)

# Convert to cohort-month granularity (using offset from config)
rx_expanded = rx_expanded.withColumn(
    "COHORT_MONTH",
    F.date_format(assign_cohort_month(F.col("TXN_DATE")), "yyyy-MM-dd")
)
print("Shape :", rx_expanded.count(), len(rx_expanded.columns))
rx_expanded.filter(F.col("PATIENT_ID") == "590252630").orderBy("COHORT_MONTH").display()

In [0]:
# ===============================
# Step 6: Aggregate per patient per month
# ===============================
final_df = (
    rx_expanded
    .groupBy("PATIENT_ID", "COHORT_MONTH")
    .agg(
        F.collect_set("Treatment_Category").alias("TREATMENT_CATEGORY"),
        F.collect_set("CATEGORY").alias("CATEGORY")
    )
    .orderBy("PATIENT_ID", "COHORT_MONTH")
)

# Join cohort labels (C1-C22); rows before C1 get null label (lookback)
final_df = final_df.join(
    df_cohort_ref.select(
        F.date_format("cohort_ref_month", "yyyy-MM-dd").alias("_ref_month_str"),
        "cohort_label"
    ),
    final_df.COHORT_MONTH == F.col("_ref_month_str"),
    "left"
).drop("_ref_month_str")

print("Shape :", final_df.count(), len(final_df.columns))
final_df.filter(F.col("PATIENT_ID") == "590252630").orderBy("COHORT_MONTH").display()

In [0]:
df_rx_clean = final_df

In [0]:
df_rx_clean = df_rx_clean.withColumnRenamed("COHORT_MONTH", "DATE")

In [0]:
# DATE is already in yyyy-MM-dd format from assign_cohort_month
df_rx_clean = df_rx_clean.withColumn("DATE", F.to_date(F.col("DATE"), "yyyy-MM-dd"))

In [0]:
print("Shape :", df_rx_clean.count(), len(df_rx_clean.columns))

In [0]:
# VALIDATION: write to a separate path so we don't overwrite production data
df_rx_clean.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/kerendia_nba_3_1_rx_journey_validation")